In [ ]:
from qiskit import QuantumCircuit
import numpy as np
import re

def build_qft(n):
    """
    Builds an n-qubit QFT circuit from scratch.
    Follows the standard textbook construction: Hadamard + controlled phase rotations.
    I'm not using Qiskit's built-in QFT here because I want full control over the gate labels.
    """
    qc = QuantumCircuit(n)
    
    for t in range(n):
        # hadamard on the current qubit to kick off the superposition
        qc.h(t)
        
        # now apply controlled rotations from all the qubits below this one
        for c in range(t + 1, n):
            k = c - t + 1  # this gives us R2, R3, R4, etc.
            angle = np.pi / (2**(k-1))
            qc.cp(angle, c, t, label='R' + str(k))
        
        # barrier just keeps the diagram from rearranging things visually
        qc.barrier()
    
    # QFT outputs bits in reversed order, so we need swaps to fix that
    # for 4 qubits: swap qubit 0<->3 and 1<->2
    qc.swap(0,3)
    qc.swap(1,2)
    
    return qc

def relabel(fig):
    """
    Qiskit labels controlled phase gates as 'P(pi/x)' by default which is ugly.
    This just goes through and replaces them with the nicer Rk notation for the report.
    """
    ax = fig.axes[0]
    
    for text_obj in ax.texts:
        content = text_obj.get_text()
        
        # look for the phase gate labels on the target side
        if content.startswith('P ('):
            m = re.search(r'/(\d+)', content)  # grab the denominator
            if m:
                denom = int(m.group(1))
                k = int(np.log2(denom)) + 1  # work backwards to get the Rk index
                text_obj.set_text('R' + str(k))
    
    return fig

# --- actually build and draw the circuit ---
qc = build_qft(4)  # 4 qubit QFT for our analysis

fig = qc.draw(
    output='mpl',
    fold=-1,               # don't wrap the circuit, keep it on one line
    style={'name': 'bw'},  # black and white looks cleaner for the report
    plot_barriers=False     # hide the barriers in the final diagram
)

# fix up the gate labels before displaying
relabel(fig)